# Agentic Translation QA - Dataset EDA

본 노트북은 병렬 코퍼스를 분석하여 Agentic Translation QA 평가용 소규모 벤치마크 셋을 설계하기 위한 EDA를 수행한다.  
주요 목표는 데이터 구조와 분포를 파악하고, 특정 도메인의 핵심 용어를 포함하는 30-50문장 평가셋을 선정하며, 최종 리포트용 시각화를 생성하는 것이다.

In [ ]:
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib

plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 200)

OUTPUT_DIR = Path("../outputs")
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 평가 데이터셋 선정 기준

- 평가 문장 수: 30-50개  
- 각 문장은 한국어 source와 영어 reference를 포함  
- 특정 도메인의 핵심 용어 5개 이상 포함  
- 길이/표현 편향 최소화  
- Baseline vs Agent-RAG 비교 실험에 재사용 가능

In [ ]:
# 기본: 프로젝트에 이미 있는 eval_set 사용
data_path = Path("../data/datasets/eval_set.json")
df = pd.read_json(data_path, encoding="utf-8")
print(df.shape)
df.head(3)

In [ ]:
print(df.columns.tolist())
print("\nmissing:\n", df.isna().sum())

# list/dict 컬럼(예: key_terms)은 해시 불가하므로 중복 검사는 텍스트 컬럼 기준으로 수행
if {"source_text", "reference"}.issubset(df.columns):
    dup_count = df.duplicated(subset=["source_text", "reference"]).sum()
else:
    hashable_cols = [c for c in df.columns if not df[c].apply(lambda v: isinstance(v, (list, dict))).any()]
    dup_count = df[hashable_cols].duplicated().sum() if hashable_cols else 0

print("\nduplicated (hashable subset):", dup_count)
df.sample(min(5, len(df)), random_state=42)

In [ ]:
COLUMN_MAP = {
    "source_text": "source_ko",
    "reference": "reference_en",
}

df = df.rename(columns=COLUMN_MAP).copy()
for col in ["source_ko", "reference_en"]:
    df[col] = df[col].astype(str).str.strip()

df = df.replace({"": np.nan}).dropna(subset=["source_ko", "reference_en"])
df = df.drop_duplicates(subset=["source_ko", "reference_en"]).reset_index(drop=True)
print(df.shape)

In [ ]:
def simple_tokenize(text: str):
    return re.findall(r"\w+", text.lower())

df["char_len_ko"] = df["source_ko"].str.len()
df["char_len_en"] = df["reference_en"].str.len()
df["token_len_ko"] = df["source_ko"].apply(lambda x: len(simple_tokenize(x)))
df["token_len_en"] = df["reference_en"].apply(lambda x: len(simple_tokenize(x)))
df["length_ratio"] = df["char_len_en"] / df["char_len_ko"].replace(0, np.nan)
df["has_number"] = df["source_ko"].str.contains(r"\d", regex=True)
df["special_char_ratio"] = df["source_ko"].apply(
    lambda x: len(re.findall(r"[^\w\s가-힣]", x)) / max(len(x), 1)
)
df[["source_ko", "reference_en", "char_len_ko", "char_len_en", "length_ratio"]].head()

전처리 결과, 분석 대상은 한국어 source와 영어 reference가 모두 존재하는 병렬 문장으로 정리되었다.  
이후 EDA에서는 문장 길이, 길이 비율, 도메인 용어 포함 여부를 중심으로 평가셋 후보군을 구성한다.

In [ ]:
summary_stats = df[["char_len_ko", "char_len_en", "token_len_ko", "token_len_en", "length_ratio"]].describe().T
summary_stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["char_len_ko"], bins=20, kde=True, ax=axes[0], color="royalblue")
axes[0].set_title("Korean Sentence Length Distribution")
axes[0].set_xlabel("Character Length")

sns.histplot(df["char_len_en"], bins=20, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("English Sentence Length Distribution")
axes[1].set_xlabel("Character Length")

plt.tight_layout()
plt.savefig(FIG_DIR / "01_length_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(
    data=df.sample(min(len(df), 3000), random_state=42),
    x="char_len_ko",
    y="char_len_en",
    alpha=0.5,
    s=30,
)
plt.title("KO-EN Sentence Length Correlation")
plt.xlabel("Korean Character Length")
plt.ylabel("English Character Length")
plt.tight_layout()
plt.savefig(FIG_DIR / "02_length_correlation.png", dpi=200, bbox_inches="tight")
plt.show()

## 도메인 핵심 용어 정의

본 예시는 법률 도메인을 기본으로 사용한다.

- 계약
- 조항
- 책임
- 의무
- 손해배상
- 위반
- 합의

In [ ]:
DOMAIN_TERMS = ["계약", "조항", "책임", "의무", "손해배상", "위반", "합의"]

def find_terms(text: str, terms: list[str]) -> list[str]:
    return [term for term in terms if term in text]

df["matched_terms"] = df["source_ko"].apply(lambda x: find_terms(x, DOMAIN_TERMS))
df["matched_term_count"] = df["matched_terms"].apply(len)
df["contains_domain_term"] = df["matched_term_count"] > 0
df[["source_ko", "matched_terms", "matched_term_count"]].head(10)

In [ ]:
term_counter = Counter()
for terms in df["matched_terms"]:
    term_counter.update(terms)

term_df = pd.DataFrame(term_counter.items(), columns=["term", "count"]).sort_values("count", ascending=False)
term_label_map = {
    "계약": "contract",
    "조항": "clause",
    "책임": "liability",
    "의무": "obligation",
    "손해배상": "damages",
    "위반": "breach",
    "합의": "agreement",
}
term_df["term_display"] = term_df["term"].map(term_label_map).fillna(term_df["term"])

plt.figure(figsize=(8, 5))
sns.barplot(data=term_df, x="count", y="term_display", color="steelblue")
plt.title("Domain Term Frequency")
plt.xlabel("Count")
plt.ylabel("Term")
plt.tight_layout()
plt.savefig(FIG_DIR / "03_domain_term_frequency.png", dpi=200, bbox_inches="tight")
plt.show()

term_df

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="matched_term_count", color="seagreen")
plt.title("Matched Domain Terms per Sample")
plt.xlabel("Number of Matched Terms")
plt.ylabel("Sample Count")
plt.tight_layout()
plt.savefig(FIG_DIR / "04_terms_per_sample.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
candidate_df = df[
    (df["matched_term_count"] >= 1)
    & (df["char_len_ko"].between(15, 120))
    & (df["char_len_en"].between(10, 180))
].copy()

print(candidate_df.shape)
candidate_df.head()

In [ ]:
if len(candidate_df) == 0:
    selected_eval_df = df.sample(min(40, len(df)), random_state=42).copy()
else:
    bins = min(5, candidate_df["char_len_ko"].nunique())
    if bins >= 2:
        candidate_df["length_bin"] = pd.qcut(candidate_df["char_len_ko"], q=bins, duplicates="drop")
        selected_eval_df = (
            candidate_df.groupby("length_bin", group_keys=False)
            .apply(lambda x: x.sample(min(len(x), 8), random_state=42))
            .reset_index(drop=True)
        )
    else:
        selected_eval_df = candidate_df.sample(min(40, len(candidate_df)), random_state=42)

selected_eval_df = selected_eval_df.head(40).copy()
selected_eval_df["is_selected"] = True

print(selected_eval_df.shape)
selected_eval_df[["source_ko", "reference_en", "matched_terms", "char_len_ko"]].head(10)

In [ ]:
compare_df = pd.concat(
    [candidate_df.assign(group="candidate"), selected_eval_df.assign(group="selected")],
    ignore_index=True,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(
    data=compare_df,
    x="char_len_ko",
    hue="group",
    stat="density",
    common_norm=False,
    bins=20,
    ax=axes[0],
)
axes[0].set_title("Candidate vs Selected: KO Length")

sns.histplot(
    data=compare_df,
    x="matched_term_count",
    hue="group",
    stat="density",
    common_norm=False,
    discrete=True,
    ax=axes[1],
)
axes[1].set_title("Candidate vs Selected: Domain Term Density")

plt.tight_layout()
plt.savefig(FIG_DIR / "05_candidate_vs_selected.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
selected_eval_df["difficulty_note"] = np.select(
    [
        selected_eval_df["matched_term_count"] >= 3,
        selected_eval_df["char_len_ko"] >= selected_eval_df["char_len_ko"].median(),
    ],
    ["high_term_density", "long_sentence"],
    default="standard",
)

sample_table = selected_eval_df[
    ["source_ko", "reference_en", "matched_terms", "matched_term_count", "char_len_ko", "difficulty_note"]
].head(10)

sample_table

## EDA 해석 및 평가셋 선정 근거

원천 병렬 코퍼스는 source-reference 구조가 안정적이며, 선택한 도메인 용어가 실제 문장에 반복적으로 등장한다.  
후보군과 최종 선택셋 분포를 비교했을 때 길이/용어 밀도 측면의 대표성을 유지하도록 구성했기 때문에, 특정 패턴에 치우치지 않은 벤치마크 셋으로 활용할 수 있다.

In [ ]:
selected_eval_df.to_csv(OUTPUT_DIR / "selected_eval_set.csv", index=False, encoding="utf-8-sig")
summary_stats.to_csv(OUTPUT_DIR / "summary_stats.csv", encoding="utf-8-sig")
term_df.to_csv(OUTPUT_DIR / "domain_term_frequency.csv", index=False, encoding="utf-8-sig")

print("Saved:")
print(OUTPUT_DIR / "selected_eval_set.csv")
print(FIG_DIR)

## 후속 실험 연결

본 노트북에서 생성한 `selected_eval_set.csv`는 이후 공통 입력으로 사용한다.

- Baseline 번역 결과 생성
- Agent-RAG 번역 결과 생성
- 용어 정확도 및 수정률 계산
- Agent 재시도 횟수/응답 시간 집계
- 오류 유형 분류 및 성공/실패 사례 분석